# Day 1 — DistilGPT-2 Inference Benchmark

This notebook runs `distilbert/distilgpt2` and records basic single-request inference metrics.

## Metrics collected

- Model loading time
- Model parameter count
- Estimated parameter memory
- Input and generated token counts
- End-to-end generation latency
- Output tokens per second
- Process RAM before and after model loading
- Process RAM before, after, and at the sampled peak during inference
- CUDA allocated and peak allocated memory when a GPU is available

The first generation is used only as a warm-up. It is not included in the measured results.

## 1. Install dependencies

Run this cell in Google Colab. On a local Jupyter environment, it installs the same required packages.

In [ ]:
%pip install -q transformers accelerate psutil pandas

## 2. Imports and configuration

For the first run, keep `MAX_NEW_TOKENS = 25` and `NUMBER_OF_RUNS = 3`. The notebook automatically uses a GPU when Colab provides one; otherwise it uses CPU.

In [ ]:
from __future__ import annotations

import gc
import os
import platform
import statistics
import threading
import time
from dataclasses import asdict, dataclass

import pandas as pd
import psutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "distilbert/distilgpt2"
PROMPT = "Artificial intelligence infrastructure is important because"
MAX_NEW_TOKENS = 25
NUMBER_OF_RUNS = 3
WARMUP_NEW_TOKENS = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Selected device: {DEVICE}")

## 3. Inspect the runtime before loading the model

This reports the remote Colab runtime when used in Colab. It does not consume your computer's main RAM for model inference.

In [ ]:
process = psutil.Process(os.getpid())
system_memory = psutil.virtual_memory()

environment_info = {
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "operating_system": platform.platform(),
    "device": str(DEVICE),
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_name": (
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    ),
    "total_system_ram_gb": system_memory.total / 1024**3,
    "available_system_ram_gb": system_memory.available / 1024**3,
    "python_process_ram_mb": process.memory_info().rss / 1024**2,
}

pd.DataFrame(
    environment_info.items(), columns=["Metric", "Value"]
)

## 4. Load DistilGPT-2 and measure loading cost

`low_cpu_mem_usage=True` reduces unnecessary peak CPU memory during model loading. The model is placed on the selected device after loading.

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

ram_before_load_mb = process.memory_info().rss / 1024**2
load_start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    low_cpu_mem_usage=True,
)
model.to(DEVICE)
model.eval()

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

model_load_seconds = time.perf_counter() - load_start
ram_after_load_mb = process.memory_info().rss / 1024**2

parameter_count = sum(parameter.numel() for parameter in model.parameters())
parameter_bytes = sum(
    parameter.numel() * parameter.element_size()
    for parameter in model.parameters()
)

model_info = {
    "model_name": MODEL_NAME,
    "model_load_seconds": model_load_seconds,
    "parameter_count": parameter_count,
    "estimated_parameter_memory_mb": parameter_bytes / 1024**2,
    "process_ram_before_load_mb": ram_before_load_mb,
    "process_ram_after_load_mb": ram_after_load_mb,
    "process_ram_load_delta_mb": ram_after_load_mb - ram_before_load_mb,
}

pd.DataFrame(model_info.items(), columns=["Metric", "Value"])

## 5. Prepare the prompt

The tokenizer converts the prompt into token IDs. DistilGPT-2 is a decoder-only model, so `generate()` returns the original prompt tokens followed by newly generated tokens.

In [ ]:
encoded_input = tokenizer(PROMPT, return_tensors="pt")
encoded_input = {
    name: tensor.to(DEVICE)
    for name, tensor in encoded_input.items()
}

input_token_count = encoded_input["input_ids"].shape[1]

print("Prompt:", PROMPT)
print("Input token count:", input_token_count)
print("Input token IDs:", encoded_input["input_ids"][0].tolist())

## 6. Define memory sampling and benchmark helpers

A lightweight background sampler checks the Python process RAM during generation. Its peak value is an approximation because sampling occurs at short intervals.

In [ ]:
@dataclass
class RunResult:
    run_number: int
    input_tokens: int
    output_tokens: int
    latency_seconds: float
    output_tokens_per_second: float
    process_ram_before_mb: float
    process_ram_after_mb: float
    sampled_peak_process_ram_mb: float
    process_ram_delta_mb: float
    cuda_allocated_before_mb: float | None
    cuda_allocated_after_mb: float | None
    cuda_peak_allocated_mb: float | None


class ProcessMemorySampler:
    def __init__(self, interval_seconds: float = 0.01) -> None:
        self.interval_seconds = interval_seconds
        self.peak_rss_bytes = 0
        self._stop_event = threading.Event()
        self._thread: threading.Thread | None = None

    def _sample(self) -> None:
        sampled_process = psutil.Process(os.getpid())
        while not self._stop_event.is_set():
            rss_bytes = sampled_process.memory_info().rss
            self.peak_rss_bytes = max(self.peak_rss_bytes, rss_bytes)
            time.sleep(self.interval_seconds)

    def start(self) -> None:
        self.peak_rss_bytes = psutil.Process(os.getpid()).memory_info().rss
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._sample, daemon=True)
        self._thread.start()

    def stop(self) -> float:
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join()
        return self.peak_rss_bytes / 1024**2


def synchronize_device() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def run_generation(run_number: int, max_new_tokens: int) -> tuple[RunResult, str]:
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    ram_before_mb = process.memory_info().rss / 1024**2
    cuda_before_mb = (
        torch.cuda.memory_allocated() / 1024**2
        if DEVICE.type == "cuda"
        else None
    )

    memory_sampler = ProcessMemorySampler()
    memory_sampler.start()

    synchronize_device()
    start_time = time.perf_counter()

    with torch.inference_mode():
        output_ids = model.generate(
            **encoded_input,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    synchronize_device()
    latency_seconds = time.perf_counter() - start_time
    sampled_peak_ram_mb = memory_sampler.stop()

    ram_after_mb = process.memory_info().rss / 1024**2
    cuda_after_mb = (
        torch.cuda.memory_allocated() / 1024**2
        if DEVICE.type == "cuda"
        else None
    )
    cuda_peak_mb = (
        torch.cuda.max_memory_allocated() / 1024**2
        if DEVICE.type == "cuda"
        else None
    )

    output_token_count = output_ids.shape[1] - input_token_count
    output_tokens_per_second = output_token_count / latency_seconds

    result = RunResult(
        run_number=run_number,
        input_tokens=input_token_count,
        output_tokens=output_token_count,
        latency_seconds=latency_seconds,
        output_tokens_per_second=output_tokens_per_second,
        process_ram_before_mb=ram_before_mb,
        process_ram_after_mb=ram_after_mb,
        sampled_peak_process_ram_mb=sampled_peak_ram_mb,
        process_ram_delta_mb=ram_after_mb - ram_before_mb,
        cuda_allocated_before_mb=cuda_before_mb,
        cuda_allocated_after_mb=cuda_after_mb,
        cuda_peak_allocated_mb=cuda_peak_mb,
    )

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return result, generated_text

## 7. Warm-up run

The warm-up initializes execution paths and is intentionally excluded from the final benchmark table.

In [ ]:
warmup_result, warmup_text = run_generation(
    run_number=0,
    max_new_tokens=WARMUP_NEW_TOKENS,
)

print(f"Warm-up latency: {warmup_result.latency_seconds:.4f} seconds")
print(f"Warm-up output tokens: {warmup_result.output_tokens}")
print("Warm-up completed.")

## 8. Run the measured benchmark

In [ ]:
results: list[RunResult] = []
last_generated_text = ""

for run_number in range(1, NUMBER_OF_RUNS + 1):
    result, generated_text = run_generation(
        run_number=run_number,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    results.append(result)
    last_generated_text = generated_text

    print(
        f"Run {run_number}: "
        f"{result.latency_seconds:.4f} s, "
        f"{result.output_tokens_per_second:.2f} output tokens/s, "
        f"peak process RAM {result.sampled_peak_process_ram_mb:.2f} MB"
    )

## 9. View all recorded inference metrics

In [ ]:
results_df = pd.DataFrame(asdict(result) for result in results)
results_df.round(4)

## 10. View the benchmark summary

In [ ]:
latencies = [result.latency_seconds for result in results]
throughputs = [result.output_tokens_per_second for result in results]
peak_ram_values = [result.sampled_peak_process_ram_mb for result in results]

summary = {
    "model": MODEL_NAME,
    "device": str(DEVICE),
    "prompt": PROMPT,
    "input_tokens": input_token_count,
    "configured_max_new_tokens": MAX_NEW_TOKENS,
    "measured_runs": NUMBER_OF_RUNS,
    "mean_latency_seconds": statistics.mean(latencies),
    "median_latency_seconds": statistics.median(latencies),
    "min_latency_seconds": min(latencies),
    "max_latency_seconds": max(latencies),
    "mean_output_tokens_per_second": statistics.mean(throughputs),
    "median_output_tokens_per_second": statistics.median(throughputs),
    "highest_sampled_process_ram_mb": max(peak_ram_values),
    "model_load_seconds": model_load_seconds,
}

summary_df = pd.DataFrame(summary.items(), columns=["Metric", "Value"])
summary_df

## 11. Inspect the generated text

DistilGPT-2 is a small English-language base model rather than an instruction-following chatbot, so repetition or low-quality continuation is normal.

In [ ]:
print(last_generated_text)

## 12. Produce text for `docs/day-01-notes.md`

Run this cell, copy its output, and paste it into the Day 1 notes file in the repository.

In [ ]:
notes_text = f"""# Day 1 — Basic DistilGPT-2 Inference Benchmark

## Environment

- Device: {DEVICE}
- CUDA device: {environment_info['cuda_device_name']}
- Python: {environment_info['python_version']}
- PyTorch: {environment_info['pytorch_version']}
- Total runtime RAM: {environment_info['total_system_ram_gb']:.2f} GB
- Available runtime RAM before model loading: {environment_info['available_system_ram_gb']:.2f} GB

## Model

- Model: {MODEL_NAME}
- Parameters: {parameter_count:,}
- Estimated parameter memory: {parameter_bytes / 1024**2:.2f} MB
- Model loading time: {model_load_seconds:.4f} seconds
- Process RAM increase during loading: {ram_after_load_mb - ram_before_load_mb:.2f} MB

## Inference configuration

- Prompt: `{PROMPT}`
- Input tokens: {input_token_count}
- Maximum new tokens: {MAX_NEW_TOKENS}
- Decoding: greedy (`do_sample=False`)
- Warm-up runs: 1
- Measured runs: {NUMBER_OF_RUNS}

## Results

| Metric | Value |
|---|---:|
| Mean latency | {statistics.mean(latencies):.4f} s |
| Median latency | {statistics.median(latencies):.4f} s |
| Minimum latency | {min(latencies):.4f} s |
| Maximum latency | {max(latencies):.4f} s |
| Mean output throughput | {statistics.mean(throughputs):.4f} tokens/s |
| Highest sampled process RAM | {max(peak_ram_values):.2f} MB |

## Generated text

```text
{last_generated_text}
```

## Initial observations

- This experiment measured one request at a time.
- The reported latency covers the complete `model.generate()` call.
- Output throughput was calculated as generated output tokens divided by total generation time.
- The warm-up run was excluded from the measured results.
- Process peak RAM was sampled periodically and is therefore an approximation.
- This experiment did not measure TTFT, TPOT, prefill time, decode time, concurrent requests, or KV-cache behavior.
"""

print(notes_text)

## 13. Optional cleanup

Use this only when you are finished and want to release model memory within the current notebook session.

In [ ]:
# Uncomment and run when finished:
# del model
# del tokenizer
# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
# print("Model memory released as far as the runtime allows.")